# Notebook 6: Survival Analysis

This notebook evaluates the prognostic significance of spatial transcriptomic features using Kaplan-Meier survival analysis, with validation in the TCGA dataset.

**Overview:**
1. Load BICRC clinical metadata and spatial feature scores
2. Optimal threshold selection (ROC + Youden index)
3. Kaplan-Meier curves for disease-free survival (DFS)
4. KRAS mutation stratification analysis
5. TCGA validation (POSTN expression, KRAS mutation)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_curve, auc
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from statsmodels.stats.multitest import multipletests

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

## 2. Load Clinical Metadata

In [ ]:
# Load BICRC survival metadata
# Columns: Patient, CRT_start, Final_observation, DFS_event, DFS_months, Survival_event, OS_months
df_meta = pd.read_csv('../data/bicrc_metadata.txt', sep='\t', index_col=0)
print('Metadata shape:', df_meta.shape)
print(df_meta.head())
print('\nColumns:', df_meta.columns.tolist())

In [ ]:
# Assign response groups
def get_response(patient_id):
    pt_num = int(patient_id.replace('Pt-', ''))
    if pt_num <= 8:
        return 'SD'
    elif pt_num <= 16:
        return 'MPR'
    else:
        return 'CR'

df_meta['Response'] = df_meta.index.map(get_response)
response_colors = {'SD': 'red', 'MPR': 'orange', 'CR': 'blue'}

print('\nResponse group distribution:')
print(df_meta['Response'].value_counts())

## 3. Kaplan-Meier Analysis by Response Group

In [ ]:
def plot_km_curve(df, duration_col, event_col, group_col, groups, colors, title, output_path):
    """Plot Kaplan-Meier survival curves with log-rank test p-value.
    
    Parameters
    ----------
    df : DataFrame with survival data
    duration_col : column name for time-to-event
    event_col : column name for event indicator (1=event, 0=censored)
    group_col : column name for grouping variable
    groups : ordered list of group labels
    colors : dict mapping group label to color
    """
    fig, ax = plt.subplots(figsize=(7, 5))
    kmf = KaplanMeierFitter()
    
    group_data = {}
    for group in groups:
        mask = df[group_col] == group
        if mask.sum() == 0:
            continue
        T = df.loc[mask, duration_col]
        E = df.loc[mask, event_col]
        kmf.fit(T, E, label=f'{group} (n={mask.sum()})')
        kmf.plot_survival_function(
            ax=ax, ci_show=True,
            color=colors.get(group, 'grey')
        )
        group_data[group] = (T, E)
    
    # Log-rank test
    if len(group_data) >= 2:
        group_labels = list(group_data.keys())
        T_all = pd.concat([group_data[g][0] for g in group_labels])
        E_all = pd.concat([group_data[g][1] for g in group_labels])
        labels_all = pd.concat([
            pd.Series([g] * len(group_data[g][0])) for g in group_labels
        ])
        result = multivariate_logrank_test(T_all, labels_all, event_observed=E_all)
        pval = result.p_value
        ax.text(0.65, 0.95, f'Log-rank p={pval:.4f}',
                transform=ax.transAxes, fontsize=10, va='top')
    
    ax.set_title(title)
    ax.set_xlabel('Time (months)')
    ax.set_ylabel('Survival probability')
    ax.set_ylim(0, 1.05)
    ax.legend(loc='lower left')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    return pval if len(group_data) >= 2 else None

In [ ]:
# KM curve by response group for DFS
plot_km_curve(
    df_meta,
    duration_col='DFS_months',
    event_col='DFS_event',
    group_col='Response',
    groups=['SD', 'MPR', 'CR'],
    colors=response_colors,
    title='Disease-Free Survival by Response Group (BICRC)',
    output_path='../figures/06_km_dfs_by_response.png'
)

## 4. Optimal Threshold for Spatial Feature Scores

For each spatial feature (e.g., subcluster proportion, neighborhood enrichment score), an optimal binary threshold is determined using the ROC curve and Youden index (sensitivity + specificity − 1).

In [ ]:
def find_optimal_threshold(feature_values, event_labels):
    """Find optimal binary threshold using ROC curve and Youden index.
    
    Returns the threshold maximizing sensitivity + specificity.
    """
    fpr, tpr, thresholds = roc_curve(event_labels, feature_values)
    youden_index = tpr - fpr
    optimal_idx = np.argmax(youden_index)
    optimal_threshold = thresholds[optimal_idx]
    roc_auc = auc(fpr, tpr)
    return optimal_threshold, roc_auc, fpr, tpr


def km_analysis_with_threshold(df_survival, feature_col, duration_col, event_col,
                                 feature_name, output_prefix):
    """Run KM analysis on a continuous feature using Youden-optimal threshold.
    
    Returns the log-rank p-value.
    """
    df = df_survival.dropna(subset=[feature_col, duration_col, event_col])
    
    # Find optimal threshold
    threshold, roc_auc, fpr, tpr = find_optimal_threshold(
        df[feature_col], df[event_col]
    )
    print(f'{feature_name}: threshold={threshold:.4f}, AUC={roc_auc:.4f}')
    
    # Dichotomize
    df = df.copy()
    df['group'] = np.where(df[feature_col] > threshold, 'High', 'Low')
    
    # KM plot
    p = plot_km_curve(
        df,
        duration_col=duration_col,
        event_col=event_col,
        group_col='group',
        groups=['High', 'Low'],
        colors={'High': '#d62728', 'Low': '#1f77b4'},
        title=f'DFS by {feature_name}\n(threshold={threshold:.3f}, AUC={roc_auc:.3f})',
        output_path=f'../figures/{output_prefix}_km.png'
    )
    
    # ROC plot
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr, tpr, color='navy', label=f'AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.axvline(fpr[np.argmax(tpr - fpr)], color='grey', linestyle=':', alpha=0.7)
    ax.set_xlabel('1 - Specificity')
    ax.set_ylabel('Sensitivity')
    ax.set_title(f'ROC Curve: {feature_name}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'../figures/{output_prefix}_roc.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return p, threshold, roc_auc

In [ ]:
# Load per-patient spatial feature scores (e.g., subcluster proportions at Pre-treatment)
# These are derived from the neighborhood analysis or composition analysis
try:
    df_features = pd.read_csv('../results/patient_spatial_features.csv', index_col=0)
    print('Loaded spatial features:', df_features.shape)
except FileNotFoundError:
    print('Spatial features file not found. Compute from Notebook 04 outputs.')
    df_features = None

# Merge with clinical metadata
if df_features is not None:
    df_surv = df_meta.join(df_features, how='inner')
    print('Merged survival + features:', df_surv.shape)

In [ ]:
# Run KM analysis for key spatial features
survival_results = []

if 'df_surv' in dir() and df_features is not None:
    feature_columns = [
        c for c in df_features.columns
        if c not in df_meta.columns and df_surv[c].notna().sum() >= 10
    ]
    
    for feature_col in feature_columns:
        try:
            p, threshold, roc_auc = km_analysis_with_threshold(
                df_surv, feature_col,
                duration_col='DFS_months',
                event_col='DFS_event',
                feature_name=feature_col,
                output_prefix=f'06_{feature_col.replace(" ", "_")}'
            )
            survival_results.append({
                'feature': feature_col,
                'threshold': threshold,
                'AUC': roc_auc,
                'logrank_p': p,
            })
        except Exception as e:
            print(f'Failed {feature_col}: {e}')
    
    if survival_results:
        df_survival_results = pd.DataFrame(survival_results)
        # BH correction
        df_survival_results['logrank_p_adj'] = multipletests(
            df_survival_results['logrank_p'].fillna(1.0), method='fdr_bh'
        )[1]
        df_survival_results.sort_values('logrank_p').to_csv(
            '../results/survival_analysis_results.csv', index=False
        )
        print('\nTop significant features:')
        print(df_survival_results[df_survival_results['logrank_p_adj'] < 0.05].to_string())

## 5. KRAS Mutation Stratification

In [ ]:
# Load genomic data
df_exome = pd.read_csv('../data/exome.txt', sep='\t', index_col=0)

# Merge with survival data
df_kras = df_meta.join(df_exome[['RAS']], how='left')
df_kras['KRAS_status'] = df_kras['RAS'].fillna('Unknown')

print('KRAS status distribution:')
print(df_kras['KRAS_status'].value_counts())

In [ ]:
# KM curve: KRAS mutant vs wild-type
df_kras_clean = df_kras[df_kras['KRAS_status'].isin(['Mut', 'WT'])]

if len(df_kras_clean) >= 4:
    plot_km_curve(
        df_kras_clean,
        duration_col='DFS_months',
        event_col='DFS_event',
        group_col='KRAS_status',
        groups=['Mut', 'WT'],
        colors={'Mut': '#d62728', 'WT': '#1f77b4'},
        title='Disease-Free Survival by KRAS Status (BICRC)',
        output_path='../figures/06_km_dfs_kras_status.png'
    )

## 6. TCGA Validation

Key findings are validated in the publicly available TCGA colorectal cancer cohort using POSTN expression (TPM) and KRAS mutation data.

In [ ]:
# Load TCGA POSTN expression data
try:
    df_tcga_postn_crc = pd.read_csv('../data/crc_postn.txt', sep='\t', index_col=0)
    df_tcga_postn_rect = pd.read_csv('../data/rect_postn.txt', sep='\t', index_col=0)
    print('TCGA CRC POSTN:', df_tcga_postn_crc.shape)
    print('TCGA Rectal POSTN:', df_tcga_postn_rect.shape)
except FileNotFoundError as e:
    print(f'File not found: {e}')
    df_tcga_postn_crc = df_tcga_postn_rect = None

# Load TCGA KRAS mutation data
try:
    df_tcga_kras = pd.read_csv('../data/tcga_kras_colon_rect.tsv', sep='\t', index_col=0)
    print('TCGA KRAS:', df_tcga_kras.shape)
except FileNotFoundError:
    print('TCGA KRAS file not found.')
    df_tcga_kras = None

In [ ]:
# TCGA: POSTN expression stratified by KRAS mutation status
if df_tcga_postn_rect is not None and df_tcga_kras is not None:
    # Merge POSTN expression with KRAS mutation status
    common_samples = df_tcga_postn_rect.index.intersection(df_tcga_kras.index)
    df_tcga = df_tcga_postn_rect.loc[common_samples, ['POSTN']].join(
        df_tcga_kras.loc[common_samples, 'KRAS_status']
    ).dropna()
    
    print('TCGA merged samples:', len(df_tcga))
    
    kras_groups = df_tcga['KRAS_status'].unique()
    if len(kras_groups) >= 2:
        mut_vals = df_tcga.loc[df_tcga['KRAS_status'] == 'Mutant', 'POSTN']
        wt_vals = df_tcga.loc[df_tcga['KRAS_status'] == 'Wild-type', 'POSTN']
        stat, pval = stats.mannwhitneyu(mut_vals, wt_vals, alternative='two-sided')
        print(f'TCGA POSTN: KRAS Mutant vs WT: U={stat:.0f}, p={pval:.4f}')
        
        fig, ax = plt.subplots(figsize=(5, 5))
        sns.boxplot(
            data=df_tcga, x='KRAS_status', y='POSTN',
            palette={'Mutant': '#d62728', 'Wild-type': '#1f77b4'}, ax=ax
        )
        ax.set_title(f'TCGA Rectal Cancer: POSTN Expression\nby KRAS Status (p={pval:.4f})')
        ax.set_ylabel('POSTN (TPM)')
        plt.tight_layout()
        plt.savefig('../figures/06_tcga_postn_kras.png', dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
# TCGA survival analysis: POSTN high vs low
if df_tcga_postn_rect is not None:
    # Check if survival data is included in the POSTN file
    survival_cols = [c for c in df_tcga_postn_rect.columns if 'survival' in c.lower() or 'event' in c.lower()]
    print('Potential survival columns:', survival_cols)
    
    # If survival data is available, run KM analysis
    if len(survival_cols) >= 2:
        duration_col = survival_cols[0]
        event_col = survival_cols[1]
        
        df_tcga_surv = df_tcga_postn_rect.dropna(subset=['POSTN', duration_col, event_col])
        threshold, roc_auc, fpr, tpr = find_optimal_threshold(
            df_tcga_surv['POSTN'], df_tcga_surv[event_col]
        )
        df_tcga_surv = df_tcga_surv.copy()
        df_tcga_surv['POSTN_group'] = np.where(df_tcga_surv['POSTN'] > threshold, 'High', 'Low')
        
        plot_km_curve(
            df_tcga_surv,
            duration_col=duration_col,
            event_col=event_col,
            group_col='POSTN_group',
            groups=['High', 'Low'],
            colors={'High': '#d62728', 'Low': '#1f77b4'},
            title=f'TCGA Rectal Cancer: Survival by POSTN Expression\n(threshold={threshold:.2f}, AUC={roc_auc:.3f})',
            output_path='../figures/06_tcga_km_postn.png'
        )

## 7. Summary of Significant Prognostic Features

In [ ]:
# Display summary of survival analysis results
if 'df_survival_results' in dir():
    print('=== Significant Prognostic Features (FDR < 0.05) ===')
    sig = df_survival_results[
        df_survival_results['logrank_p_adj'] < 0.05
    ].sort_values('logrank_p')
    
    if len(sig) > 0:
        print(sig[['feature', 'AUC', 'logrank_p', 'logrank_p_adj']].to_string(index=False))
    else:
        print('No features with FDR < 0.05. Top features by nominal p-value:')
        print(df_survival_results.nsmallest(10, 'logrank_p')[
            ['feature', 'AUC', 'logrank_p', 'logrank_p_adj']
        ].to_string(index=False))